# Tiny GPT Reader: pretrain on English, then learn to answer questions

Trains a ~17.5M-parameter GPT that reads word pieces, on Simple English Wikipedia. Step 2 (later) teaches it to answer questions from a passage.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. The smoke test takes about 2 minutes. The full run takes about 2 hours: keep this tab open.
3. Progress is saved to your Google Drive (`MyDrive/tiny-gpt-reader`) every few minutes. **If Colab disconnects**, run the cells again from the top: training continues where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Save progress to Google Drive
Colab asks for permission to use your Drive. The checkpoint, tokenizer and logs go to `MyDrive/tiny-gpt-reader`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/tiny-gpt-reader"

## Get the code

In [ ]:
import os
os.chdir("/content")
if os.path.exists("tiny-gpt"):
    !cd tiny-gpt && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/Nenethsu/tiny-gpt.git
os.chdir("/content/tiny-gpt")
assert os.path.exists("pretrain.py"), "pretrain.py is missing: push the latest code to GitHub first"
!python bpe.py test

## Smoke test (about 2 minutes)
A tiny model for 60 steps, to catch problems before the long run. It downloads Wikipedia once (157 MB); the full run reuses it.

In [ ]:
!python pretrain.py --preset smoke --out /content/reader-smoke

## Full pretraining (about 2 hours)
Prints the loss, the time left and sample sentences every 500 steps. Loss should fall from about 9 to around 3.5 or lower.

If Colab disconnects: run the cells from the top again (mount Drive, get the code), then this one. It resumes from the last save.

In [ ]:
!python pretrain.py --out "$OUT"

## Results

In [ ]:
import json
log = json.load(open(f"{OUT}/training.json"))
print(f"{log['params']:,} parameters, {log['steps_done']:,} steps, {log['train_seconds'] / 60:.0f} min on {log['device']}")
print(f"{log['train_tokens']:,} training tokens from {log['articles']:,} articles")
print("final loss:", log["loss"][-1])
for s in log["samples"][-1]["texts"]:
    print("-", s[:200])

# Step 2: learn to answer questions, in a conversation

Starts from the pretrained model above and fine-tunes it on SQuAD 2.0 (about 130,000 questions about Wikipedia passages; a third have no answer, so it learns to say so) and CoQA (conversations about a passage, so it learns to follow up: "How old was **she**?"). Saved to `MyDrive/tiny-gpt-reader/qa` and resumes the same way.

Two exams it never trains on: SQuAD 2.0 dev (11,873 questions) and CoQA dev (about 8,000 conversation turns, asked with their history).

## Smoke test (about 2 minutes)
Makes a tiny pretrained model first (in case this is a new session), then fine-tunes it for 40 steps.

In [ ]:
!python pretrain.py --preset smoke --out /content/reader-smoke && python finetune.py --preset smoke --init /content/reader-smoke --out /content/reader-qa-smoke

## Fine-tune (about 45-60 minutes)
Prints both exams every 1,000 steps (exact match, F1, "no answer" accuracy, and how often the safety check would show the answer), then takes the full exams.

In [ ]:
!python finetune.py --init "$OUT" --out "$OUT/qa"

## Exam results

In [ ]:
import json
qa = json.load(open(f"{OUT}/qa/training.json"))
print(json.dumps(qa["final"], indent=1))
for name in ("squad", "coqa"):
    print("
" + name.upper())
    for e in qa["examples"][name]:
        before = " / ".join(f"{q} -> {a}" for q, a in e["history"])
        print(f"  {'[' + before + '] ' if before else ''}Q: {e['question']}
     expected {e['expected'][0]!r}, got {e['got']!r}")

# Step 3: confidence cut-off and export (CPU is fine)

Works on the **CPU runtime** too (Runtime > Change runtime type > CPU), so the GPU limit doesn't matter here. Run the Drive and "Get the code" cells at the top first; the GPU check cell just prints an error on CPU, which is fine.

The cut-off decides when the chat shows the Reader's own answer: only when it's confident enough that shown answers are right at least 80% of the time. Otherwise the chat shows the source sentence.

## Score the exams with confidence
Prints, for each confidence cut-off, how often the Reader's answer would be shown, how often it's exactly right, and how often it's useful (mostly overlaps the right answer), with three rules: the cut-off alone, the cut-off plus the **agreement check** (the answer must lie inside the sentence the chat's search picks), and yes/no answers on their own.

On CPU, `--limit 1500` scores a random 1,500 questions per exam (about 10 minutes). On a GPU, remove `--limit 1500` to score all of them. Every answer is saved to `predictions.json`, so `--from-predictions` redoes the tables in seconds.

In [ ]:
!python evaluate.py --qa "$OUT/qa" --limit 1500

## Export for the portfolio
Writes the browser files and downloads `tiny-gpt-reader-web.zip` (about 18 MB). Unzip it into the portfolio's `public/models/tiny-gpt-reader/`.

In [ ]:
!python export_reader.py --root "$OUT" --out /content/reader-web
!cd /content && rm -f tiny-gpt-reader-web.zip && zip -qj tiny-gpt-reader-web.zip reader-web/*
from google.colab import files
files.download("/content/tiny-gpt-reader-web.zip")

# Step 4: the pointing Reader (span extraction, GPU)

Instead of writing its answer, the Reader points at where the answer starts and ends in the passage (or at yes / no / no answer). It reuses the pretraining from step 1; only the fine-tuning is new. Needs a **GPU** (T4). Saved to `MyDrive/tiny-gpt-reader/span` and resumes the same way.

Run the Drive and "Get the code" cells at the top first.

## Smoke test (about 2 minutes)

In [ ]:
!python pretrain.py --preset smoke --out /content/reader-smoke && python finetune_span.py --preset smoke --init /content/reader-smoke --out /content/reader-span-smoke

## Fine-tune the pointing Reader (about 25-40 minutes)
Prints both exams every 1,000 steps, then the full exams, the confidence cut-off table and the recommended cut-off.

In [ ]:
!python finetune_span.py --init "$OUT" --out "$OUT/span"

## Agreement tables (seconds, CPU is fine)
Re-reads the saved answers and adds the agreement check and the yes/no table.

In [ ]:
!python evaluate.py --qa "$OUT/span" --from-predictions